In [1]:
import pandas as pd
import akshare as ak
import numpy as np
import datetime
import xgboost
import scipy
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

In [2]:
start_date = '20150101'
end_date = '20241231'
index_zz1000 = ak.index_stock_cons_csindex(symbol="000852")
code_list  = index_zz1000['成分券代码'].to_list()
print(code_list[:5])

['000012', '000019', '000028', '000029', '000030']


In [3]:
df = ak.stock_zh_a_hist(symbol='000001', start_date=start_date, end_date=end_date, adjust="qfq")
print(df.columns)
df

Index(['日期', '股票代码', '开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌幅', '涨跌额',
       '换手率'],
      dtype='object')


,日期,股票代码,开盘,收盘,最高,最低,成交量,成交额,振幅,涨跌幅,涨跌额,换手率
0,2015-01-05,000001,7.94,7.96,8.14,7.67,2860436,4.565388e+09,5.99,1.53,0.12,2.91
1,2015-01-06,000001,7.85,7.80,8.22,7.64,2166421,3.453446e+09,7.29,-2.01,-0.16,2.20
2,2015-01-07,000001,7.64,7.59,7.83,7.46,1700121,2.634796e+09,4.74,-2.69,-0.21,1.73
3,2015-01-08,000001,7.60,7.23,7.65,7.19,1407714,2.128003e+09,6.06,-4.74,-0.36,1.43
4,2015-01-09,000001,7.19,7.31,7.86,7.05,2508500,3.835378e+09,11.20,1.11,0.08,2.55
...,...,...,...,...,...,...,...,...,...,...,...,...
2426,2024-12-25,000001,11.26,11.32,11.42,11.24,1475283,1.759957e+09,1.60,0.53,0.06,0.76
2427,2024-12-26,000001,11.32,11.26,11.33,11.18,1000075,1.183746e+09,1.33,-0.53,-0.06,0.52
2428,2024-12-27,000001,11.27,11.23,11.30,11.06,1290012,1.518383e+09,2.13,-0.27,-0.03,0.66
2429,2024-12-30,000001,11.18,11.35,11.37,11.18,1351846,1.610892e+09,1.69,1.07,0.12,0.70


In [ ]:
# 下载原始数据zz1000+zz500
save_dir = Path(r'D:\python\graduation_thesis\zz1000_parquet')
max_workers = 5
save_dir.mkdir(parents=True, exist_ok=True)

def download_and_save(symbol):
    data = ak.stock_zh_a_hist(symbol = symbol, start_date=start_date,
                              end_date=end_date,adjust='qfq')
    data = data.rename(columns={
        "日期": "date",
        "股票代码": "symbol",
        "开盘": "open",
        "收盘": "close",
        "最高": "high",
        "最低": "low",
        "成交量": "volume",
        "成交额": "amount",
        "振幅": "amplitude",
        "涨跌幅": "pct_chg",
        "涨跌额": "chg",
        "换手率": "turnover"
    })
    data['date'] = pd.to_datetime(data['date'])
    data['symbol'] = data['symbol'].astype(str).str.zfill(6)
    data = data.sort_values("date").reset_index(drop=True)
    data.to_parquet(save_dir/f'{symbol}.parquet',index=False)
failed = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_symbol = {executor.submit(download_and_save,symbol):symbol for symbol in code_list}
    for i,future in enumerate(as_completed(future_to_symbol),1):
        symbol = future_to_symbol[future]
        try:
            future.result()
            print(f'[{i}/{len(code_list)}] saved {symbol}')
        except Exception as e:
            failed.append((symbol,str(e)))
            print(f'{symbol} failed with {e}')

print(failed)

[1/1000] saved 000029
[2/1000] saved 000030
[3/1000] saved 000019
[4/1000] saved 000028
[5/1000] saved 000012
[6/1000] saved 000048
[7/1000] saved 000035
[8/1000] saved 000058
[9/1000] saved 000059
[10/1000] saved 000061
[11/1000] saved 000065
[12/1000] saved 000089
[13/1000] saved 000049
[14/1000] saved 000156
[15/1000] saved 000099
[16/1000] saved 000403
[17/1000] saved 000420
[18/1000] saved 000503
[19/1000] saved 000422
[20/1000] saved 000543
[21/1000] saved 000498
[22/1000] saved 000552
[23/1000] saved 000555
[24/1000] saved 000557
[25/1000] saved 000550
[26/1000] saved 000567
[27/1000] saved 000581
[28/1000] saved 000603
[29/1000] saved 000589
[30/1000] saved 000597
[31/1000] saved 000600
[32/1000] saved 000628
[33/1000] saved 000612
[34/1000] saved 000650
[35/1000] saved 000620
[36/1000] saved 000680
[37/1000] saved 000636
[38/1000] saved 000681
[39/1000] saved 000685
[40/1000] saved 000676
[41/1000] saved 000682
[42/1000] saved 000686
[43/1000] saved 000672
[44/1000] saved 0006

In [1]:
import pandas as pd
import numpy as np
# 检查全市场合并后数据质量
fp = r'D:\python\graduation_thesis\factor_all.parquet'
df = pd.read_parquet(fp)

print('shape:', df.shape)
print('columns[:10]:', df.columns[:10].tolist())
print('dtypes:\n', df.dtypes.head(10))
print('date min/max:', df['date'].min(), df['date'].max())
print('n_symbol:', df['symbol'].nunique())

# 1. 主键检查
dup = df.duplicated(['date', 'symbol']).sum()
print('duplicated (date, symbol):', dup)

# 2. 排序后检查单股时间是否有乱序
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
bad_order = df.groupby('symbol')['date'].apply(lambda s: not s.is_monotonic_increasing).sum()
print('symbols with non-monotonic dates:', bad_order)

# 3. 每日/每股样本量概览
cnt_by_date = df.groupby('date').size()
cnt_by_symbol = df.groupby('symbol').size()
print('rows per date describe:\n', cnt_by_date.describe())
print('rows per symbol describe:\n', cnt_by_symbol.describe())

# 4. 缺失和inf
na_rate = df.isna().mean().sort_values(ascending=False)
print('top 20 NA rate:\n', na_rate.head(20))

num_cols = df.select_dtypes(include=[np.number]).columns
inf_count = np.isinf(df[num_cols].to_numpy()).sum()
print('inf count:', inf_count)

# 5. 随机抽几列看看分布
sample_cols = [c for c in num_cols if c not in ['date']]
sample_cols = sample_cols[:10]
print(df[sample_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']])

shape: (2712234, 121)
columns[:10]: ['date', 'symbol', 'ret_1', 'ret_5', 'ret_10', 'ret_20', 'ret_sum_5', 'ret_sum_10', 'ret_sum_20', 'vol_5']
dtypes:
 date          datetime64[ns]
symbol                object
ret_1                float64
ret_5                float64
ret_10               float64
ret_20               float64
ret_sum_5            float64
ret_sum_10           float64
ret_sum_20           float64
vol_5                float64
dtype: object
date min/max: 2015-02-03 00:00:00 2024-12-31 00:00:00
n_symbol: 1485
duplicated (date, symbol): 0
symbols with non-monotonic dates: 0
rows per date describe:
 count    2397.000000
mean     1131.511890
std       246.509817
min         1.000000
25%       932.000000
50%      1149.000000
75%      1356.000000
max      1479.000000
dtype: float64
rows per symbol describe:
 count    1485.000000
mean     1826.420202
std       687.923384
min         5.000000
25%      1300.000000
50%      2234.000000
75%      2357.000000
max      2390.000000
dtype: 

In [2]:
print('rows per date describe:\n', cnt_by_date.describe())
print('rows per symbol describe:\n', cnt_by_symbol.describe())

rows per date describe:
 count    2397.000000
mean     1131.511890
std       246.509817
min         1.000000
25%       932.000000
50%      1149.000000
75%      1356.000000
max      1479.000000
dtype: float64
rows per symbol describe:
 count    1485.000000
mean     1826.420202
std       687.923384
min         5.000000
25%      1300.000000
50%      2234.000000
75%      2357.000000
max      2390.000000
dtype: float64


In [ ]:
import pandas as pd
import numpy as np
# 去除na值
factor_fp = r'D:\python\graduation_thesis\factor_all.parquet'

df = pd.read_parquet(factor_fp)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

factor_cols = [c for c in df.columns if c not in ['date', 'symbol']]

# 先把 inf 变成 NaN，再做按股票前向填充
df[factor_cols] = df[factor_cols].replace([np.inf, -np.inf], np.nan)
df[factor_cols] = df.groupby('symbol', sort=False)[factor_cols].ffill()

df.to_parquet(factor_fp, index=False)

print(df.shape)
print(df.head())
print(df.tail())

(2741894, 121)
        date  symbol     ret_1     ret_5    ret_10    ret_20  ret_sum_5  \
0 2015-02-05  000009 -2.292994 -5.074257 -3.884712 -3.400504  -5.053302   
1 2015-02-06  000009 -0.130378 -1.415701 -2.420382 -0.648508  -1.347047   
2 2015-02-09  000009 -1.827676 -1.182654 -5.764411 -1.052632  -1.115521   
3 2015-02-10  000009  1.728723 -1.797176 -4.494382 -2.671756  -1.752106   
4 2015-02-11  000009 -0.653595 -3.184713 -5.236908 -2.813299  -3.175919   

   ret_sum_10  ret_sum_20     vol_5  ...  turnover_z_20  doji_ratio_20  \
0   -3.777411   -2.856232  2.249384  ...      -0.669802           0.15   
1   -2.278716   -0.089885  1.751579  ...      -0.426495           0.15   
2   -5.762443   -0.490843  1.706122  ...      -2.112585           0.15   
3   -4.409660   -2.183172  1.522036  ...      -1.201811           0.15   
4   -5.188098   -2.327861  1.415175  ...      -1.880789           0.15   

   marubozu_ratio_20  green_ratio_20  red_ratio_20  body_sum_to_range_sum_20  \
0        

In [ ]:
#去除整体样本末尾20天
import polars as pl
factor_fp = r"D:\python\graduation_thesis\factor_all.parquet"
df = pl.read_parquet(factor_fp)
# 保证时间类型正确，并按 symbol/date 排序
df = (
    df.with_columns(
        pl.col("date").str.strptime(pl.Date, strict=False)
        if df.schema["date"] == pl.Utf8
        else pl.col("date")
    )
    .sort(["symbol", "date"])
)

factor_cols = [c for c in df.columns if c not in ["date", "symbol"]]
df = (
    df.with_columns([
        pl.int_range(pl.len()).over("symbol").alias("_idx"),
        pl.len().over("symbol").alias("_n"),
    ])
    .filter(pl.col("_idx") < pl.col("_n") - 20)
    .drop(["_idx", "_n"])
)

df.write_parquet(factor_fp)
print(df.shape)
print(df.head())

(2712234, 121)
shape: (5, 121)
┌────────────┬────────┬───────────┬───────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ date       ┆ symbol ┆ ret_1     ┆ ret_5     ┆ … ┆ amplitude_ ┆ corr_ret_ ┆ corr_ret_ ┆ corr_vol_ │
│ ---        ┆ ---    ┆ ---       ┆ ---       ┆   ┆ to_absret_ ┆ turnover_ ┆ amount_20 ┆ turnover_ │
│ datetime[n ┆ str    ┆ f64       ┆ f64       ┆   ┆ sum_20     ┆ 20        ┆ ---       ┆ 20        │
│ s]         ┆        ┆           ┆           ┆   ┆ ---        ┆ ---       ┆ f64       ┆ ---       │
│            ┆        ┆           ┆           ┆   ┆ f64        ┆ f64       ┆           ┆ f64       │
╞════════════╪════════╪═══════════╪═══════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 2015-02-05 ┆ 000009 ┆ -2.292994 ┆ -5.074257 ┆ … ┆ 1.835395   ┆ -0.026009 ┆ -0.005035 ┆ 0.999987  │
│ 00:00:00   ┆        ┆           ┆           ┆   ┆            ┆           ┆           ┆           │
│ 2015-02-06 ┆ 000009 ┆ -0.130378 ┆ -1.415701 ┆ … ┆ 1.991751

In [ ]:
#去除整体样本末尾20天
label_fp = r"D:\python\graduation_thesis\label_all.parquet"
label_fp_out = r"D:\python\graduation_thesis\label_all_filt.parquet"
df = pl.read_parquet(label_fp)

df = (
    df.with_columns(
        pl.col("date").str.strptime(pl.Date, strict=False)
        if df.schema["date"] == pl.Utf8
        else pl.col("date")
    )
    .sort(["symbol", "date"])
    .with_columns([
        pl.int_range(pl.len()).over("symbol").alias("_idx"),
        pl.len().over("symbol").alias("_n"),
    ])
    .filter(pl.col("_idx") < pl.col("_n") - 20)
    .drop(["_idx", "_n"])
)

df.write_parquet(label_fp_out)
print(df.shape)
print(df.tail())

(2712234, 11)
shape: (5, 11)
┌────────────┬────────┬───────────┬───────────┬───┬───────────┬────────────┬───────────┬───────────┐
│ date       ┆ symbol ┆ fwd_ret_1 ┆ fwd_ret_5 ┆ … ┆ fwd_vol_5 ┆ fwd_vol_20 ┆ fwd_ret_5 ┆ fwd_ret_2 │
│ ---        ┆ ---    ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---        ┆ _voladj   ┆ 0_voladj  │
│ datetime[n ┆ str    ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64        ┆ ---       ┆ ---       │
│ s]         ┆        ┆           ┆           ┆   ┆           ┆            ┆ f64       ┆ f64       │
╞════════════╪════════╪═══════════╪═══════════╪═══╪═══════════╪════════════╪═══════════╪═══════════╡
│ 2024-11-27 ┆ 689009 ┆ -0.013984 ┆ 0.02228   ┆ … ┆ 0.017015  ┆ 0.019166   ┆ 1.309476  ┆ 1.59533   │
│ 00:00:00   ┆        ┆           ┆           ┆   ┆           ┆            ┆           ┆           │
│ 2024-11-28 ┆ 689009 ┆ 0.032933  ┆ 0.038221  ┆ … ┆ 0.014604  ┆ 0.018826   ┆ 2.617224  ┆ 2.502643  │
│ 00:00:00   ┆        ┆           ┆           ┆   ┆           

In [ ]:
import polars as pl
# 检查清洗后数据的index
f = (
    pl.read_parquet(r"D:\python\graduation_thesis\factor_all.parquet")
    .select(["symbol", "date"])
    .sort(["symbol", "date"])
)

l = (
    pl.read_parquet(r"D:\python\graduation_thesis\label_all_filt.parquet")
    .select(["symbol", "date"])
    .sort(["symbol", "date"])
)

print(f.shape, l.shape)
print(f.equals(l))
f

(2712234, 2) (2712234, 2)
True


symbol,date
str,datetime[ns]
"""000009""",2015-02-05 00:00:00
"""000009""",2015-02-06 00:00:00
"""000009""",2015-02-09 00:00:00
"""000009""",2015-02-10 00:00:00
"""000009""",2015-02-11 00:00:00
…,…
"""689009""",2024-11-27 00:00:00
"""689009""",2024-11-28 00:00:00
"""689009""",2024-11-29 00:00:00


In [27]:
# 切分数据集

import polars as pl
TRAIN_END = pl.date(2022, 12, 1)
VALID_END = pl.date(2023, 12, 1)

factor_fp = r"D:\python\graduation_thesis\factor_all.parquet"
label_fp  = r"D:\python\graduation_thesis\label_all_filt.parquet"

factor = pl.read_parquet(factor_fp).sort(["symbol", "date"])
label  = pl.read_parquet(label_fp).sort(["symbol", "date"])

# factor split
factor_train = factor.filter(pl.col("date") < TRAIN_END)
factor_valid = factor.filter((pl.col("date") >= TRAIN_END) & (pl.col("date") < VALID_END))
factor_test  = factor.filter(pl.col("date") >= VALID_END)

# label split
label_train = label.filter(pl.col("date") < TRAIN_END)
label_valid = label.filter((pl.col("date") >= TRAIN_END) & (pl.col("date") < VALID_END))
label_test  = label.filter(pl.col("date") >= VALID_END)

# write
factor_train.write_parquet(r"D:\python\graduation_thesis\data\factor_train.parquet")
factor_valid.write_parquet(r"D:\python\graduation_thesis\data\factor_valid.parquet")
factor_test.write_parquet(r"D:\python\graduation_thesis\data\factor_test.parquet")

label_train.write_parquet(r"D:\python\graduation_thesis\data\label_train.parquet")
label_valid.write_parquet(r"D:\python\graduation_thesis\data\label_valid.parquet")
label_test.write_parquet(r"D:\python\graduation_thesis\data\label_test.parquet")

print(factor_train.shape, factor_valid.shape, factor_test.shape)
print(label_train.shape, label_valid.shape, label_test.shape)

(2009314, 121) (346323, 121) (356597, 121)
(2009314, 11) (346323, 11) (356597, 11)


In [29]:
import os
import polars as pl

DATA_DIR = r"D:\python\graduation_thesis\data"

FILES = [
    "label_train.parquet",
    "label_valid.parquet",
    "label_test.parquet",
]

# 只给这些“纯 ret / ewm_ret”列加 rank
RANK_COLS = [
    "fwd_ret_1",
    "fwd_ret_5",
    "fwd_ewm_ret_5",
    "fwd_ewm_ret_20",
]

DROP_COLS = [
    "fwd_ret_20",
]

DATE_COL = "date"

# True: 数值越大 rank 越靠前，最大值 rank=1
# False: 数值越小 rank 越靠前，最小值 rank=1
DESCENDING = False


def add_cross_sectional_ranks(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []

    for c in RANK_COLS:
        if c in df.columns:
            exprs.append(
                pl.col(c)
                .rank(method="ordinal", descending=DESCENDING)
                .over(DATE_COL)
                .cast(pl.Int32)
                .alias(f"{c}_rank")
            )

    if exprs:
        df = df.with_columns(exprs)

    return df


def process_one_file(fp: str) -> None:
    df = pl.read_parquet(fp)

    # 删列
    drop_exist = [c for c in DROP_COLS if c in df.columns]
    if drop_exist:
        df = df.drop(drop_exist)

    # 加按日横截面 rank
    df = add_cross_sectional_ranks(df)

    # 直接覆盖原文件
    df.write_parquet(fp)

    print(f"done: {fp}")
    print(df.columns)


def main():
    for fn in FILES:
        fp = os.path.join(DATA_DIR, fn)
        process_one_file(fp)


if __name__ == "__main__":
    main()

done: D:\python\graduation_thesis\data\label_train.parquet
['date', 'symbol', 'fwd_ret_1', 'fwd_ret_5', 'fwd_ewm_ret_5', 'fwd_ewm_ret_20', 'fwd_vol_5', 'fwd_vol_20', 'fwd_ret_5_voladj', 'fwd_ret_20_voladj', 'fwd_ret_1_rank', 'fwd_ret_5_rank', 'fwd_ewm_ret_5_rank', 'fwd_ewm_ret_20_rank']
done: D:\python\graduation_thesis\data\label_valid.parquet
['date', 'symbol', 'fwd_ret_1', 'fwd_ret_5', 'fwd_ewm_ret_5', 'fwd_ewm_ret_20', 'fwd_vol_5', 'fwd_vol_20', 'fwd_ret_5_voladj', 'fwd_ret_20_voladj', 'fwd_ret_1_rank', 'fwd_ret_5_rank', 'fwd_ewm_ret_5_rank', 'fwd_ewm_ret_20_rank']
done: D:\python\graduation_thesis\data\label_test.parquet
['date', 'symbol', 'fwd_ret_1', 'fwd_ret_5', 'fwd_ewm_ret_5', 'fwd_ewm_ret_20', 'fwd_vol_5', 'fwd_vol_20', 'fwd_ret_5_voladj', 'fwd_ret_20_voladj', 'fwd_ret_1_rank', 'fwd_ret_5_rank', 'fwd_ewm_ret_5_rank', 'fwd_ewm_ret_20_rank']


In [36]:
import polars as pl
import pandas as pd
import datetime
df = pd.read_parquet(r"D:\python\graduation_thesis\data\label_test.parquet")
df2 = df[['date','symbol','fwd_ret_1']]
df2 = pd.read_parquet(r"D:\python\graduation_thesis\predict_return\fwd_ret_1\daily_perf.parquet")
df2

,date,capital_start,gross_return,turnover,trading_cost,net_return,capital_end,n_universe,n_long,n_short,cum_nav,drawdown
0,2023-12-01,1.000000e+08,0.001412,0.500000,0.000500,0.000912,1.000912e+08,1455,72,72,1.000912,0.000000
1,2023-12-04,1.000912e+08,-0.001359,0.125000,0.000125,-0.001484,9.994269e+07,1455,72,72,0.999427,-0.001484
2,2023-12-05,9.994269e+07,0.004112,0.138889,0.000139,0.003973,1.003397e+08,1455,72,72,1.003397,0.000000
3,2023-12-06,1.003397e+08,0.002729,0.173611,0.000174,0.002556,1.005962e+08,1454,72,72,1.005962,0.000000
4,2023-12-07,1.005962e+08,-0.004412,0.125000,0.000125,-0.004537,1.001397e+08,1453,72,72,1.001397,-0.004537
...,...,...,...,...,...,...,...,...,...,...,...,...
238,2024-11-27,1.540703e+08,-0.001076,0.191781,0.000192,-0.001268,1.538749e+08,1478,73,73,1.538749,-0.009329
239,2024-11-28,1.538749e+08,-0.003967,0.198630,0.000199,-0.004165,1.532340e+08,1477,73,73,1.532340,-0.013455
240,2024-11-29,1.532340e+08,0.000734,0.239726,0.000240,0.000494,1.533097e+08,1479,73,73,1.533097,-0.012967
241,2024-12-02,1.533097e+08,0.010865,0.260274,0.000260,0.010604,1.549355e+08,1479,73,73,1.549355,-0.002500
